# 02 — Baselines

**Owner:** Person A · **Course session:** S1 (Foundations)

Loads the frozen split from Notebook 01 and scores two baselines through the shared
`src.eval.metrics.evaluate` — the same function every later notebook uses, so every table in
this project is comparable.

- **Random** — the floor. If a real model can't beat this, it isn't learning anything.
- **Popularity** — the real baseline to beat. On a head-concentrated catalog like Electronics
  this is a *strong* baseline on Recall/NDCG, and that is the interesting part.

In [1]:
!git clone https://github.com/21zaimotman-tech/amazon-recsys.git
%cd amazon-recsys
!pip install -q -r requirements.txt

fatal: destination path 'amazon-recsys' already exists and is not an empty directory.
/content/amazon-recsys


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, ".." if Path("../src").exists() else ".")

import pandas as pd

from src import config as C
from src.baselines.popularity import PopularityRecommender, RandomRecommender
from src.data.split import ground_truth_from
from src.eval.metrics import evaluate

DATA = Path("../data") if Path("../data").exists() else Path("data")
train = pd.read_parquet(DATA / "train.parquet")
test = pd.read_parquet(DATA / "test.parquet")
print(f"train={len(train):,}  test={len(test):,}")

train=111,481  test=13,116


## Eval setup

- **Catalog** = distinct items seen in train (the recommendable universe).
- **Ground truth** = test-period positives (`rating >= 4`), one set per user.
- **Exclusion** = each user's train-seen items are removed from their candidate list before
  scoring, so a model can't get credit for re-recommending something the user already has.

In [3]:
catalog = train.item_id.nunique()
test_gt = ground_truth_from(test, positive_only=True)
users = list(test_gt.keys())
seen = train.groupby("user_id")["item_id"].agg(set).to_dict()
N = max(C.K_VALUES)

print(f"catalog={catalog:,}  test users scored={len(users):,}  k_values={C.K_VALUES}")

catalog=9,487  test users scored=5,009  k_values=(10, 20, 50)


In [4]:
rows = []
for name, Model in [("Random", RandomRecommender), ("Popularity", PopularityRecommender)]:
    model = Model().fit(train)
    recs = model.recommend(users, n=N, exclude=seen)
    m = evaluate(recs, test_gt, catalog_size=catalog, k_values=C.K_VALUES)
    m["method"] = name
    rows.append(m)

baseline_results = pd.DataFrame(rows).set_index("method")
report_cols = [c for c in baseline_results.columns if c != "n_users_scored"]
baseline_results[report_cols].round(4)

,Recall@10,NDCG@10,Recall@20,NDCG@20,Recall@50,NDCG@50,Coverage@50
method,,,,,,,
Random,0.0011,0.0007,0.0024,0.0011,0.0049,0.0016,1.0000
Popularity,0.0122,0.0071,0.0175,0.0088,0.0499,0.0164,0.0066


## Interpretation

**Popularity's Recall/NDCG is far above Random's, but its Coverage is close to zero** (it
recommends the same handful of head items to every user, so across all users it only ever
touches a tiny slice of the catalog) **while Random's Coverage is ~1.0** (given enough users,
a uniform-random pick eventually samples almost the whole catalog, it just never lands on the
*right* item for a given user). That's the required contrast for `ANALYSIS.md` §1: Popularity
wins on accuracy exactly because Electronics reviews concentrate on a small set of items, and
loses completely on diversity for the same reason. Any retriever we build next (MF-BPR,
two-tower) has to beat Popularity's Recall/NDCG *and* do meaningfully better than its Coverage
to be worth serving — beating Popularity on Recall alone by memorizing the head is not a win.

These two rows are the first two entries in the final comparison table (`ANALYSIS.md` §1);
Notebook 03 appends MF-BPR to it.

In [5]:
baseline_results[report_cols].round(4).to_csv(DATA / "baseline_results.csv")
print("saved -> data/baseline_results.csv")

saved -> data/baseline_results.csv
